In [1]:
from utils.load_results import *
from utils.plot_helpers import *
from utils.analysis_from_interaction import *

import pandas as pd
import seaborn as sns
from matplotlib import pyplot as plt
plt.style.use('default')
import torch
from utils.analysis_from_interaction import *
from language_analysis_local import TopographicSimilarityConceptLevel, encode_target_concepts_for_topsim
import os
if not os.path.exists('analysis'):
    os.makedirs('analysis')
#import plotly.express as px
from collections import Counter

### Utilities

In [3]:
def objects_to_concepts(sender_input, n_values):
    """reconstruct concepts from objects in interaction"""
    n_targets = int(sender_input.shape[1]/2)
    # get target objects and fixed vectors to re-construct concepts
    target_objects = sender_input[:, :n_targets]
    target_objects = k_hot_to_attributes(target_objects, n_values)
    # concepts are defined by a list of target objects (here one sampled target object) and a fixed vector
    (objects, fixed) = retrieve_concepts_sampling(target_objects, all_targets=True)
    concepts = list(zip(objects, fixed))
    return concepts

In [4]:
def retrieve_messages(interaction):
    """retrieve messages from interaction"""
    messages = interaction.message.argmax(dim=-1)
    messages = [msg.tolist() for msg in messages]
    return messages

In [5]:
def count_symbols(messages):
    """counts symbols in messages"""
    all_symbols = [symbol for message in messages for symbol in message]
    symbol_counts = Counter(all_symbols)
    return symbol_counts

In [6]:
def get_unique_message_set(messages):
    """returns unique messages as a set ready for set operations"""
    return set(tuple(message) for message in messages)

In [7]:
def get_unique_concept_set(concepts):
    """returns unique concepts"""
    concept_tuples = []
    for objects, fixed in concepts:
        tuple_objects = []
        for object in objects:
            tuple_objects.append(tuple(object))
        tuple_objects = tuple(tuple_objects)
        tuple_concept = (tuple_objects, tuple(fixed))
        concept_tuples.append(tuple_concept)
    tuple(concept_tuples)
    unique_concepts = set(concept_tuples)
    return unique_concepts

In [8]:
# new
def retrieve_messages_from_interaction(interaction, strip_eos_for_mi=True, eos_id=0, max_len_attr=3):
    # Raw
    msgs = interaction.message.argmax(dim=-1).tolist()
    # Keep EOS for display; provide a stripped version for MI
    msgs_display = [m for m in msgs]
    if strip_eos_for_mi:
        msgs_mi = []
        for m in msgs:
            mm = [t for t in m if t != eos_id]
            # optionally clip to num attributes
            msgs_mi.append(mm[:max_len_attr])
    else:
        msgs_mi = msgs_display
    return msgs_display, msgs_mi

def trim_after_eos(message, eos_id=0):
    try:
        return message[:message.index(eos_id)]
    except ValueError:
        return message


In [16]:
it = torch.load('results/3dshapes/shapes3d_feat_rep_game_size_10_vsf_3/standard/zero_shot/specific/0/interactions/train/epoch_300/interaction_gpu0', weights_only=False)
print(type(it))
print(it)
#print([k for k in it.keys()])
print(it.sender_input.shape)
print(hasattr(it, 'receiver_input'), getattr(it, 'receiver_input', None) is not None)

<class 'egg.core.interaction.Interaction'>
Interaction(sender_input=tensor([[[   0.0000, 2067.0664, 4572.2051,  ...,  722.6777,    0.0000,
          1013.9728],
         [ 953.7012,    0.0000, 1522.8708,  ...,  475.5476, 3275.8755,
          1098.2847],
         [2652.4299,    0.0000, 5362.9707,  ..., 1397.7684, 2953.9751,
          1088.2965],
         ...,
         [ 261.8389,    0.0000,    0.0000,  ...,   53.4650, 1404.5110,
          1564.5881],
         [2025.2010, 1206.0707, 5253.6475,  ...,  344.9642,    0.0000,
           193.6267],
         [2560.8918,    0.0000, 5484.1987,  ...,   44.3652, 3172.0923,
           409.6460]],

        [[2068.1616,    0.0000, 1055.9304,  ..., 1156.1156, 3867.9358,
          2253.9624],
         [3198.1360,    0.0000, 3774.0864,  ...,    0.0000, 2742.9072,
             0.0000],
         [   0.0000, 2206.0195, 3857.5605,  ...,  151.5049,    0.0000,
           570.1258],
         ...,
         [3088.8247,    0.0000, 2431.1135,  ...,  707.1614, 3629.

### Configurations

In [9]:
datasets = ['3dshapes']
n_values = [4]
n_attributes = [3]
n_epochs = 300
n_datasets = len(datasets)

path = [
    'results/3dshapes/shapes3d_feat_rep_game_size_10_vsf_3',  # path for 3dshapes
]

In [76]:
# datasets = ['3dshapes', '(3,4)']
# n_values = [4, 4]
# n_attributes = [3, 3]
# n_epochs = 300
# n_datasets = len(datasets)

# paths = [
#     'results/3dshapes/shapes3d_feat_rep_game_size_10_vsf_3',  # path for 3dshapes
#     'results/(3,4)_game_size_10_vsf_3'                        # path for symbolic (3,4)
# ]

In [10]:
context_unaware = False # whether original or context_unaware simulations are evaluated
zero_shot = True # whether zero-shot simulations are evaluated
zero_shot_test = 'specific' # 'generic' or 'specific'
test_interactions = True # whether scores should be calculated on test interactions (only with zero shot)
test_as = 'test' # 'test' or 'test_sampled_unscaled' or 'test_unscaled' or 'test_fine' 
setting = ""
if context_unaware:
    setting = setting + 'context_unaware'
else:
    setting = setting + 'standard'
if zero_shot:
    setting = setting + '/zero_shot/' + zero_shot_test

### Determine vocab size and message reuse

In [78]:
# go through all datasets
for i, d in enumerate(datasets):
    print(d)
    for run in range(3): # for 3 runs (0,1,2) per dataset because otherwise I would get an error
    #for run in range(5): --- IGNORE ---
        path_to_run = paths[i] + '/' + str(setting) +'/' + str(run) + '/'
        path_to_interaction_train = (path_to_run + 'interactions/train/epoch_' + str(n_epochs) + '/interaction_gpu0')
        path_to_interaction_val = (path_to_run + 'interactions/validation/epoch_' + str(n_epochs) + '/interaction_gpu0')
        path_to_interaction_test = (path_to_run + 'interactions/' + str(test_as) +'/epoch_0/interaction_gpu0')
        interaction_train = torch.load(path_to_interaction_train, weights_only=False)
        interaction_val = torch.load(path_to_interaction_val, weights_only=False)
        interaction_test = torch.load(path_to_interaction_test, weights_only=False)
        
        concepts_train = objects_to_concepts(interaction_train.sender_input, n_values=n_values[i])
        concepts_val = objects_to_concepts(interaction_val.sender_input, n_values=n_values[i])
        concepts_test = objects_to_concepts(interaction_test.sender_input, n_values=n_values[i])
        
        messages_train = retrieve_messages(interaction_train)
        messages_val = retrieve_messages(interaction_val)
        messages_test = retrieve_messages(interaction_test)
    
        symbol_counts_train = count_symbols(messages_train)
        symbol_counts_val = count_symbols(messages_val)
        symbol_counts_test = count_symbols(messages_test)
        symbol_counts = [symbol_counts_train, symbol_counts_val, symbol_counts_test]
        pickle.dump(symbol_counts, open(path_to_run + 'symbol_counts_' + str(test_as) + '.pkl', 'wb'))

        actual_vocab_size = len(symbol_counts_train + symbol_counts_val + symbol_counts_test)
        print(actual_vocab_size, "symbols from the alphabet have been actually used during training, validation and testing.")
        pickle.dump(actual_vocab_size, open(path_to_run + 'vocab_size_' + str(test_as) + '.pkl', 'wb'))
        
        # consider train and validation messages together
        messages_train_val = messages_train +  messages_val
        # consider only unique messages
        messages_train_val_unique = get_unique_message_set(messages_train_val)
        #print("messages train val", len(messages_train_val), len(messages_train_val_unique))
        messages_test_unique = get_unique_message_set(messages_test)
        #print("messages test", len(messages_test), len(messages_test_unique))
        # total messages
        messages_total = messages_train_val +  messages_test
        messages_total_unique = get_unique_message_set(messages_total)
        
        # concepts
        concepts_train_unique = get_unique_concept_set(concepts_train)
        concepts_val_unique = get_unique_concept_set(concepts_val)
        concepts_test_unique = get_unique_concept_set(concepts_test)
        #print("concepts", len(concepts_test), len(concepts_test_unique))
        concepts_total = concepts_train + concepts_val + concepts_test
        concepts_total_unique = get_unique_concept_set(concepts_total)
        num_of_concepts = [len(concepts_train_unique), len(concepts_val_unique), len(concepts_test_unique), len(concepts_total_unique), len(concepts_total)]
        pickle.dump(num_of_concepts, open(path_to_run + 'num_of_concepts_' + str(test_as) + '.pkl', 'wb'))
        
        # messages reused in testing:
        intersection = messages_train_val_unique & messages_test_unique
        
        # messages only used in training:
        difference_train = messages_train_val_unique - messages_test_unique
        
        # messages only used in testing:
        difference_test = messages_test_unique - messages_train_val_unique
        print(len(difference_test), "novel messages used for the", len(concepts_test_unique), "novel concepts")
        
        message_reuse = [len(intersection), len(difference_train), len(difference_test), len(concepts_test_unique), (len(difference_test)/len(concepts_test_unique)), len(messages_test_unique)]
        pickle.dump(message_reuse, open(path_to_run + 'message_reuse_' + str(test_as) + '.pkl', 'wb'))

3dshapes
16 symbols from the alphabet have been actually used during training, validation and testing.
0 novel messages used for the 600 novel concepts
16 symbols from the alphabet have been actually used during training, validation and testing.
3 novel messages used for the 600 novel concepts
15 symbols from the alphabet have been actually used during training, validation and testing.
0 novel messages used for the 600 novel concepts


very low novelty compared to (3,4) dataset results -> agents did not produce any new messages for novel concepts. Every test message was already seen in training/validation except for in run 1, with only 3 novel messages.
Not really sure what else it could mean :)

I could not adjust the code such that it would also compute symbol use and novel messages for the (3,4) symbolic dataset, because I did not find the interaction files anywhere in either of the repositories. However, here's a copy of the findings published in the public repo of zero-shot paper: 

(3,4)

16 symbols from the alphabet have been actually used during training, validation and testing.

31 novel messages used for the 64 novel concepts

15 symbols from the alphabet have been actually used during training, validation and testing.

19 novel messages used for the 64 novel concepts

14 symbols from the alphabet have been actually used during training, validation and testing.

19 novel messages used for the 64 novel concepts

15 symbols from the alphabet have been actually used during training, validation and testing.

28 novel messages used for the 64 novel concepts

16 symbols from the alphabet have been actually used during training, validation and testing.

28 novel messages used for the 64 novel concepts

In [79]:
message_reuse_dict = {'intersection': [], 'difference train': [], 'difference test': [], 'concepts test unique': [], 'test ratio': [], 'messages test unique': [],
                      'reuse rate': [], 'novelty rate': [], 'total ratio': []}
for i, d in enumerate(datasets):
    intersection, train_difference, test_difference, test_concepts, test_ratio, test_messages, reuse_rate, novelty_rate, total_ratio = [], [], [], [], [], [], [], [], []
    for run in range(3): # for 3 runs (0,1,2) per dataset because otherwise I would get an error
    #for run in range(5): --- IGNORE ---
        path_to_run = paths[i] + '/' + str(setting) +'/' + str(run) + '/'
        message_reuse = pickle.load(open(path_to_run + 'message_reuse_' + str(test_as) + '.pkl', 'rb'))
        intersection.append(message_reuse[0])
        train_difference.append(message_reuse[1])
        test_difference.append(message_reuse[2])
        test_concepts.append(message_reuse[3])
        test_ratio.append(message_reuse[4])
        test_messages.append(message_reuse[5])
        reuse_rate.append(message_reuse[0]/message_reuse[5])
        novelty_rate.append(message_reuse[2]/message_reuse[5])
        total_ratio.append(message_reuse[5]/message_reuse[3]) # test_messages / test_concepts (novel unique messages & concepts)

    message_reuse_dict['intersection'].append(intersection)
    message_reuse_dict['difference train'].append(train_difference)
    message_reuse_dict['difference test'].append(test_difference)
    message_reuse_dict['concepts test unique'].append(test_concepts)
    message_reuse_dict['test ratio'].append(test_ratio)
    message_reuse_dict['messages test unique'].append(test_messages)
    message_reuse_dict['reuse rate'].append(reuse_rate)
    message_reuse_dict['novelty rate'].append(novelty_rate)
    message_reuse_dict['total ratio'].append(total_ratio)

In [80]:
message_reuse = [message_reuse_dict['concepts test unique'], message_reuse_dict['messages test unique'], message_reuse_dict['total ratio'], message_reuse_dict['reuse rate'], message_reuse_dict['novelty rate']]

# Convert the list to a NumPy array
mess_reuse_array = np.array(message_reuse)

# Compute means and standard deviations over the five runs
means = np.mean(mess_reuse_array, axis=-1)
std_devs = np.std(mess_reuse_array, axis=-1)

# Row names and column names
# row_names = ["D(3,4)", "D(3,8)", "D(3,16)", "D(4,4)", "D(4,8)", "D(5,4)"]
row_names = ["3dshapes"]
col_names = ["test concepts", "total unique messages", "message-concept ratio", "reuse rate","novelty rate"]

# Prepare the data for the DataFrames
data = []

# iterate over datasets
for i in range(means.shape[1]):
    row = []
    # iterate over conditions
    for j in range(means.shape[0]):
        if j > 1:
            formatted_value = f"{means[j, i]:.2f} $\\pm$ {std_devs[j, i]:.2f}"
        elif j == 0:
            formatted_value = f"{int(means[j, i])}"
        else:
            formatted_value = f"{means[j, i]:.1f} $\\pm$ {std_devs[j, i]:.1f}"
        row.append(formatted_value)
    data.append(row)

# Create DataFrames
df = pd.DataFrame(data, index=row_names, columns=col_names)

# Convert DataFrames to LaTeX tables
latex_table = df.to_latex(index=True, escape=False)
print(df)
print(latex_table)

         test concepts total unique messages message-concept ratio  \
3dshapes           600         7.7 $\pm$ 4.2       0.01 $\pm$ 0.01   

               reuse rate     novelty rate  
3dshapes  0.92 $\pm$ 0.12  0.08 $\pm$ 0.12  
\begin{tabular}{llllll}
\toprule
 & test concepts & total unique messages & message-concept ratio & reuse rate & novelty rate \\
\midrule
3dshapes & 600 & 7.7 $\pm$ 4.2 & 0.01 $\pm$ 0.01 & 0.92 $\pm$ 0.12 & 0.08 $\pm$ 0.12 \\
\bottomrule
\end{tabular}



here's what we have from the (3,4) dataset from the zero-shot paper:

| Feature                   | 3DShapes | D(3,4) |                                          |
| :------------------------ | :------- | :----- | :-------------------------------------------------- |
| **Concepts**              | 600      | 12     |                |
| **Unique messages**       | ~8       | ~12    |  |
| **Message–concept ratio** | 0.01     | 0.98   |           |
| **Reuse rate**            | 0.92     | 0.80   |               |
| **Novelty rate**          | 0.08     | 0.20   |             |


message-concept ration for 3dshapes is sooo low while it's so high for the symbolic dataset

reuse rate is way higher for 3dshapes (0.92>0.80)

Novelty rate is way lower for 3dshapes (0.08<0.20)


### Symbol reuse
Also in "to generic" condition, all symbols are reused during testing, i.e. they all encode relevant information. This is why a qualitative analysis of messages makes more sense.

In [84]:
def symbol_frequency(interaction, n_attributes, n_values, vocab_size, is_gumbel=True):
    messages = interaction.message.argmax(dim=-1) if is_gumbel else interaction.message
    messages = messages[:, :-1] # without EOS
    sender_input = interaction.sender_input
    n_objects = sender_input.shape[1]
    n_targets = int(n_objects / 2)
    # k_hots = sender_input[:, :-n_attributes]
    # objects = k_hot_to_attributes(k_hots, n_values)
    target_objects = sender_input[:, :n_targets]
    target_objects = k_hot_to_attributes(target_objects, n_values)
    # intentions = sender_input[:, -n_attributes:]  # (0=same, 1=any)
    (objects, fixed) = retrieve_concepts_sampling(target_objects)

    objects[fixed == 1] = np.nan

    objects = objects
    messages = messages
    favorite_symbol = {}
    mutual_information = {}
    for att in range(n_attributes):
        for val in range(n_values):
            object_labels = (objects[:, att] == val).astype(int)
            max_MI = 0
            for symbol in range(vocab_size):
                symbol_indices = np.argwhere(messages == symbol)[0]
                symbol_labels = np.zeros(len(messages))
                symbol_labels[symbol_indices] = 1
                MI = normalized_mutual_info_score(symbol_labels, object_labels)
                if MI > max_MI:
                    max_MI = MI
                    max_symbol = symbol
            favorite_symbol[str(att) + str(val)] = max_symbol
            mutual_information[str(att) + str(val)] = max_MI

    return favorite_symbol, mutual_information

In [85]:
context_unaware = False # whether original or context_unaware simulations are evaluated
zero_shot = True # whether zero-shot simulations are evaluated
zero_shot_test = 'generic' # 'generic' or 'specific'
test_interactions = True # whether scores should be calculated on test interactions (only with zero shot)
setting = ""
if context_unaware:
    setting = setting + 'context_unaware'
else:
    setting = setting + 'standard'
if zero_shot:
    setting = setting + '/zero_shot/' + zero_shot_test


In [83]:
for run in range(3): # for 3 runs (0,1,2) per dataset because otherwise I would get an error
#for run in range(5): --- IGNORE ---
    path_to_run = paths[0] + '/' + str(setting) +'/' + str(run) + '/'
    path_to_interaction_train = (path_to_run + 'interactions/train/epoch_' + str(n_epochs) + '/interaction_gpu0')
    path_to_interaction_val = (path_to_run + 'interactions/validation/epoch_' + str(n_epochs) + '/interaction_gpu0')
    path_to_interaction_test = (path_to_run + 'interactions/test/epoch_0/interaction_gpu0')
    interaction_train = torch.load(path_to_interaction_train, weights_only=False)
    interaction_val = torch.load(path_to_interaction_val, weights_only=False)
    interaction_test = torch.load(path_to_interaction_test, weights_only=False)
    
    # retrieve "lexicon" based on mutual information
    # hard-code for D(3,4) for now
    favorite_symbol, mutual_information = symbol_frequency(interaction_train, n_attributes=3, n_values=4, vocab_size=13)
    print(favorite_symbol)

    messages = interaction_test.message.argmax(dim=-1)
    messages = [msg.tolist() for msg in messages]
    sender_input = interaction_test.sender_input
    print(sender_input.shape)
    n_targets = int(sender_input.shape[1]/2)
    # get target objects and fixed vectors to re-construct concepts
    target_objects = sender_input[:, :n_targets]
    target_objects = k_hot_to_attributes(target_objects, n_values[i])
    # concepts are defined by a list of target objects (here one sampled target object) and a fixed vector
    (objects, fixed) = retrieve_concepts_sampling(target_objects, all_targets=True)
    concepts = list(zip(objects, fixed))
    
    # DEBUG: check concepts before filtering
    print("Number of concepts:", len(concepts))
    print("Fixed vectors sum:", [sum(t_fixed) for _, t_fixed in concepts])
    # get distractor objects to re-construct context conditions
    distractor_objects = sender_input[:, n_targets:]
    distractor_objects = k_hot_to_attributes(distractor_objects, n_values[i])
    context_conds = retrieve_context_condition(objects, fixed, distractor_objects)

    # get random qualitative samples
    #fixed_index = random.randint(0, n_attributes[i]-1) # define a fixed index for the concept
    #n_fixed = random.randint(1, n_attributes[i]) # how many fixed attributes?
    n_fixed = 3
    #fixed_indices = random.sample(range(0, n_attributes[i]), k=n_fixed) # select which attributes are fixed
    fixed_indices = [0, 1, 2]
    #fixed_value = random.randint(0, n_values[i]-1) # define a fixed value for this index
    fixed_values = random.choices(range(0, n_values[i]), k=n_fixed)
    fixed_values = [3, 0, 1]
    print(n_fixed, fixed_indices, fixed_values)
    #index_threshold = 20000 # optional: define some index threshold to make sure that examples are not taken from the beginning of training
    # TODO: adapt this loop such that multiple indices can be fixed
    all_for_this_concept = []
    for idx, (t_objects, t_fixed) in enumerate(concepts):
        #if sum(t_fixed) == 1 and t_fixed[fixed_index] == 1:# and idx > index_threshold:
        if sum(t_fixed) == n_fixed and all(t_fixed[fixed_index] == 1 for fixed_index in fixed_indices):
            for t_object in t_objects:
                if all(t_object[fixed_index] == fixed_values[j] for j, fixed_index in enumerate(fixed_indices)):
                    all_for_this_concept.append((idx, t_object, t_fixed, context_conds[idx], messages[idx]))
                    fixed = t_fixed
    if len(all_for_this_concept) > 0:
        #sample = random.sample(all_for_this_concept, 20)
        sample = all_for_this_concept
        column_names = ['game_nr', 'object', 'fixed indices', 'context condition', 'message']
        df = pd.DataFrame(sample, columns=column_names)
        print(df)
        #df.to_csv('analysis/quali_' + str(d) + '_' + str(setting) + '_' + str(sample[0][1]) + ',' + str(fixed) + 'all.csv', index=False)
        #print('saved ' + 'analysis/quali_' + str(d) + '_' + str(setting) + '_' + str(sample[0][1]) + ',' + str(fixed) + 'all.csv')
    else:
        raise ValueError("sample for dataset " + str(d) + " could not be generated")

{'00': 2, '01': 3, '02': 2, '03': 12, '10': 5, '11': 8, '12': 2, '13': 10, '20': 4, '21': 0, '22': 11, '23': 9}
torch.Size([600, 20, 100])
Number of concepts: 600
Fixed vectors sum: [10.0, 11.0, 15.0, 10.0, 11.0, 7.0, 12.0, 12.0, 11.0, 15.0, 12.0, 12.0, 12.0, 11.0, 9.0, 10.0, 8.0, 12.0, 9.0, 10.0, 8.0, 13.0, 11.0, 10.0, 10.0, 8.0, 10.0, 11.0, 9.0, 12.0, 6.0, 6.0, 7.0, 1.0, 5.0, 5.0, 6.0, 4.0, 5.0, 8.0, 2.0, 2.0, 1.0, 6.0, 4.0, 5.0, 6.0, 2.0, 2.0, 6.0, 2.0, 4.0, 3.0, 7.0, 5.0, 4.0, 1.0, 2.0, 3.0, 8.0, 5.0, 4.0, 11.0, 5.0, 11.0, 8.0, 7.0, 9.0, 5.0, 9.0, 6.0, 5.0, 7.0, 7.0, 11.0, 5.0, 11.0, 10.0, 6.0, 15.0, 7.0, 5.0, 4.0, 5.0, 4.0, 4.0, 6.0, 4.0, 4.0, 7.0, 5.0, 7.0, 9.0, 8.0, 5.0, 7.0, 2.0, 4.0, 6.0, 4.0, 8.0, 2.0, 8.0, 5.0, 4.0, 7.0, 7.0, 3.0, 5.0, 1.0, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 1.0, 0.0, 0.0, 0.0, 1.0, 4.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 2.0, 2.0, 3.0, 1.0, 2.0, 2.0, 1.0, 2.0, 3.0, 3.0, 3.0, 1.0, 3.0, 3.0, 4.0, 1.0, 3.0, 3.0, 2.0, 3.0, 2.0, 2.0, 1.0, 2.0, 2.0, 2.0, 3.0, 2.0

ValueError: sample for dataset 3dshapes could not be generated

In [ ]:
# From Zero shot 
# go through all datasets
for i, d in enumerate(datasets):
    print(d)
    # get random qualitative samples
    # to specific: all indices should be fixed
    if zero_shot_test == 'specific':
        n_fixed = n_attributes[i]
        fixed_indices = list(range(0, n_attributes[i])) # all attributes fixed
        fixed_values = random.choices(range(0, n_values[i]), k=n_fixed) # define fixed values for these indices
    # to generic: only one index fixed, which one is randomly determined
    elif zero_shot_test == 'generic':
        n_fixed = 1
        fixed_indices = random.sample(range(0, n_attributes[i]), k=n_fixed) # select which attribute is fixed
        fixed_values = random.choices(range(0, n_values[i]), k=n_fixed) # define a fixed value for this index
    #print(fixed_indices, fixed_values)
    for run in range(3):
        path_to_run = paths[i] + '/' + str(setting) +'/' + str(run) + '/'
        path_to_interaction_train = (path_to_run + 'interactions/train/epoch_' + str(n_epochs) + '/interaction_gpu0')
        path_to_interaction_val = (path_to_run + 'interactions/validation/epoch_' + str(n_epochs) + '/interaction_gpu0')
        path_to_interaction_test = (path_to_run + 'interactions/' + str(test_as) + '/epoch_0/interaction_gpu0')
        interaction_train = torch.load(path_to_interaction_train, weights_only=False)
        interaction_val = torch.load(path_to_interaction_val, weights_only=False)
        interaction_test = torch.load(path_to_interaction_test, weights_only=False)
        
        # retrieve mapping between attribute-value pairs and symbols
        favorite_symbol, mutual_information = symbol_frequency_MI(interaction_train, n_attributes=n_attributes[i], n_values=n_values[i], vocab_size=vocab_sizes[i])
        print(favorite_symbol, mutual_information)
        
        messages = interaction_test.message.argmax(dim=-1)
        messages = [msg.tolist() for msg in messages]
        sender_input = interaction_test.sender_input
        n_targets = int(sender_input.shape[1]/2)
        # get target objects and fixed vectors to re-construct concepts
        target_objects = sender_input[:, :n_targets]
        target_objects = k_hot_to_attributes(target_objects, n_values[i])
        # concepts are defined by a list of target objects (here one sampled target object) and a fixed vector
        (objects, fixed) = retrieve_concepts_sampling(target_objects, all_targets=True)
        concepts = list(zip(objects, fixed))
        
        # get distractor objects to re-construct context conditions
        distractor_objects = sender_input[:, n_targets:]
        distractor_objects = k_hot_to_attributes(distractor_objects, n_values[i])
        context_conds = retrieve_context_condition(objects, fixed, distractor_objects)
        
        all_for_this_concept = []
        for idx, (t_objects, t_fixed) in enumerate(concepts):
            if sum(t_fixed) == n_fixed and all(t_fixed[fixed_index] == 1 for fixed_index in fixed_indices):
                for t_object in t_objects:
                    if all(t_object[fixed_index] == fixed_values[j] for j, fixed_index in enumerate(fixed_indices)):
                        all_for_this_concept.append((idx, t_object, t_fixed, context_conds[idx], messages[idx]))
                        fixed = t_fixed
        if len(all_for_this_concept) > 0:
            #sample = random.sample(all_for_this_concept, 20)
            sample = all_for_this_concept
            column_names = ['game_nr', 'object', 'fixed indices', 'context condition', 'message']
            sample_df = pd.DataFrame(sample, columns=column_names)
            # find out which messages have been used how often (once in test dataset test_sampled_unscaled)
            message_counts = sample_df.message.apply(tuple).value_counts()/10 # divide by game size because above single objects are taken, but we are interested in concepts (i.e. sets of game_size=10 objects)
            messages = message_counts.index.tolist()
            counts = message_counts.values.tolist()
            cond_indices = np.arange(0, len(messages)*10, 10)
            context_conds = sample_df['context condition'][cond_indices]
            used_symbols = look_up_values(fixed_indices, fixed_values, favorite_symbol)
            symbol_MI = look_up_values(fixed_indices, fixed_values, mutual_information)
            df_concept = pd.DataFrame({
                'fixed indices': [fixed_indices], 
                'fixed values': [fixed_values]
            }) 
            df_messages = pd.DataFrame({
                'context condition': context_conds.values.tolist(),
                'message': messages, 
                'counts': counts
            })
            df_symbols = pd.DataFrame({
                'symbols': used_symbols, 
                'symbol MI': symbol_MI
            })
            df = pd.concat([df_concept, df_messages, df_symbols])
            print(df.to_latex(float_format=lambda x: '{:,.0f}'.format(x) if x % 1 == 0 else '{:,.4f}'.format(x)))
            # df.to_csv('analysis/' + str(zero_shot_test) + '/quali_' + str(d) + '_' + str(zero_shot_test) + '_' + str(run) + '_' + str(fixed_indices) + ',' + str(fixed_values) + 'message_symbol_counts.csv', index=False)
            # print('saved ' + 'analysis/' + str(zero_shot_test) + '/quali_' + str(d) + '_' + str(zero_shot_test) + '_' + str(run) + '_' + str(fixed_indices) + ',' + str(fixed_values) + 'message_symbol_counts.csv')
        else:
            raise ValueError("sample for dataset " + str(d) + " could not be generated")

3dshapes


NameError: name 'symbol_frequency_MI' is not defined